# 🧭 Experiment 02: 3-Method Direction of Arrival (AoA) Benchmark
**Project:** `pynq-sound-localizer` (`v1.3.1`)  
**Target:** Production Angular Harvester & Metrological Error Characterization  
**Outputs:** `data/angle_characterization_raw.xlsx` & `data/angle_characterization_raw.csv`

---

### 🏛️ Physical Operating Principles & Angular Benchmark
This experiment runs an interferometric benchmark across **three independent mathematical methods**:

1. **TDOA Leading-Edge Threshold Delay ($\theta_{\text{TDOA}}$):**
   $$\Delta t = t_1 - t_2, \qquad \Delta r = c(T) \cdot \Delta t, \qquad \theta_{\text{TDOA}} = \arcsin\left(\text{clip}\left(\frac{\Delta r_{\text{corr}}}{d}, -1.0, 1.0\right)\right)$$
2. **Direct-Gate Autodyne Phase Interferometry ($\theta_{\text{phase}}$):**
   $$\Delta\phi_{\text{direct}} = \frac{2\pi f_0}{c(T)} \Delta r_{\text{corr}}, \qquad \theta_{\text{phase}} = \arcsin\left(\text{clip}\left(\frac{c(T) \cdot \Delta\phi_{\text{direct}}}{2\pi f_0 d}, -1.0, 1.0\right)\right)$$
3. **Normalized Differential Acoustic Energy ($\theta_{\text{energy}}$):**
   $$\frac{E_2 - E_1}{E_1 + E_2} \approx \frac{d}{r} \sin(\theta) \implies \theta_{\text{energy}} = \arcsin\left(\text{clip}\left(\frac{r}{d} \cdot \left(\frac{E_2 - E_1}{E_1 + E_2}\right), -1.0, 1.0\right)\right)$$

> **Microphone Geometry:**
> * Center origin $(0, 0)$ is the baseline midpoint.
> * **Mic 1 (A0):** Left side at $x = -2.5\,\text{cm}$.
> * **Mic 2 (A1):** Right side at $x = +2.5\,\text{cm}$ ($d = 5.0\,\text{cm}$ baseline).
> * $\theta > 0^\circ \implies$ Right sector (closer to Mic 2); $\theta < 0^\circ \implies$ Left sector (closer to Mic 1).

In [ ]:
# Cell 1: Environment Setup & Hardware Overlay Initialization
import time
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pynq_localizer import (
    MicrophoneArrayOverlay,
    KinematicAnalytics,
    AcousticProfile
)

# 1. Initialize Hardware Overlay (M=1 Bypass Mode, 500 kSPS undecimated stream)
ol = MicrophoneArrayOverlay(version="v1.5.2-rc4")
dma = ol.axi_dma_0
if not dma.recvchannel.running:
    dma.mmio.write(0x30, 0x04)
    time.sleep(0.005)
    dma.recvchannel.start()

# 2. Ambient Physical Parameters & Baseline Geometry
temperature_c = 20.0
c_sound = KinematicAnalytics.speed_of_sound(temperature_c)
mic_distance_m = 0.05  # 5.0 cm baseline
r_fixed_cm = 30.0      # Radial distance on protractor arc

# Load calibrated profile parameters
profile_path = Path("../../profiles/active_buzzer_profile.json").resolve()
if not profile_path.exists():
    profile_path = Path("profiles/active_buzzer_profile.json").resolve()

if profile_path.exists():
    profile = AcousticProfile.from_json(profile_path)
    f0 = float(profile.frequencies[0])
    cal_off1_ms = profile.calibrated_toa_offset_m1_ms
    cal_off2_ms = profile.calibrated_toa_offset_m2_ms
    ka1 = float(profile.k_values[0])
    ka2 = float(profile.k_values_m2[0])
    print(f"✅ Loaded calibrated profile: '{profile.name}' (f0 = {f0:.1f} Hz)")
else:
    f0 = KinematicAnalytics.DEFAULT_F0_HZ
    cal_off1_ms = KinematicAnalytics.DEFAULT_TOA_OFFSET_MS
    cal_off2_ms = KinematicAnalytics.DEFAULT_TOA_OFFSET_MS
    ka1, ka2 = 0.0196, 0.0196
    print(f"ℹ️ Using default parameters: f0 = {f0:.1f} Hz, offset = {cal_off1_ms:.4f} ms")

d_max = (c_sound / (2.0 * f0)) * 100.0  # Spatial aliasing limit (~6.45 cm)

print("=" * 80)
print("✅ 3-METHOD DIRECTION OF ARRIVAL (AoA) HARVESTER READY")
print("=" * 80)
print(f"• Sampling Rate      : {ol.fs_per_ch:.0f} SPS (0.00 µs inter-channel skew)")
print(f"• Carrier Frequency  : f0 = {f0:.1f} Hz")
print(f"• Speed of Sound     : c(T) = {c_sound:.2f} m/s (at {temperature_c}°C)")
print(f"• Baseline Spacing   : d = {mic_distance_m*100:.1f} cm (Spatial aliasing limit: {d_max:.2f} cm)")
print(f"• Radial Standoff Arc: r = {r_fixed_cm:.1f} cm")
print("=" * 80)

## 1. Cardinal Positions Sanity Check
Quickly test the 3 cardinal positions before running the full protractor sweep:
* **CENTER ($0^\circ$):** Directly in front at $r \approx 30\,\text{cm} \implies \Delta r \approx 0,\, \theta \approx 0^\circ$.
* **RIGHT ($+45^\circ$):** Closer to Mic 2 (A1) $\implies \Delta r > 0,\, \theta > 0$.
* **LEFT ($-45^\circ$):** Closer to Mic 1 (A0) $\implies \Delta r < 0,\, \theta < 0$.

In [ ]:
# Cell 2: 3-Position Quick Direction Check
positions = [
    ("CENTER (Broadside 0°)", "≈ 0.0°"),
    ("RIGHT (+45° mark, closer to Mic 2)", "Positive (+35° to +50°)"),
    ("LEFT (-45° mark, closer to Mic 1)", "Negative (-35° to -50°)")
]

for label, expected in positions:
    input(f"👉 Place buzzer at {label} (~30 cm away) and press [Enter]...")
    
    res = ol.capture_pulsed_toa_frame(
        pulse_width_ms=10.0,
        packet_samples=16384,
        f_target=f0,
        mic_distance_m=mic_distance_m,
        calibrated_offset_m1_ms=cal_off1_ms,
        calibrated_offset_m2_ms=cal_off2_ms,
        temperature_c=temperature_c,
        timeout=1.0
    )
    
    print(f"   • Direct Phase Bearing : {res['theta_phase_deg']:+5.1f}° (Expected: {expected})")
    print(f"   • TDOA Flight Bearing  : {res['theta_tdoa_deg']:+5.1f}°")
    print(f"   • Energy Bearing       : {res['theta_energy_deg']:+5.1f}°")
    print(f"   • Path Delta (Δr)      : {res['delta_r_cm']:+5.1f} cm")
    print(f"   • Status               : {res['status']} | SMR = {res['smr_a0_db']:.1f} dB\n")

## 2. Guided Multi-Angle Protractor Sweep ($-60^\circ$ to $+60^\circ$)
Place the protractor centered at $(0, 0)$ with a radius of $r = 30.0\,\text{cm}$.  
For each marked angle station:
1. Move the buzzer to the marked angle on the arc and press **`[Enter]`**.
2. The system fires $N=15$ pulses, logging **TDOA**, **direct autodyne phase**, and **normalized energy differences** simultaneously.
3. It reconstructs near-field Cartesian coordinates $(x, y)$ and computes statistical jitter ($\sigma$) and errors vs. protractor ground truth.

In [ ]:
# Cell 3: Production 3-Method Protractor Harvester Loop
stations_deg = [-60.0, -45.0, -30.0, -15.0, 0.0, 15.0, 30.0, 45.0, 60.0]
n_shots_per_station = 15

raw_records = []
station_summaries = []

print("=" * 85)
print(f"🚀 LAUNCHING ANGULAR HARVESTER ({len(stations_deg)} stations × {n_shots_per_station} shots = {len(stations_deg)*n_shots_per_station} pings)")
print(f"   Radius = {r_fixed_cm:.1f} cm | Baseline d = {mic_distance_m*100:.1f} cm")
print("=" * 85)

for s_idx, th_tgt in enumerate(stations_deg):
    input(f"\n🧭 [{s_idx+1}/{len(stations_deg)}] Place buzzer at {th_tgt:+5.1f}° on the {r_fixed_cm:.0f} cm arc and press [Enter]...")
    
    shot_phase = []
    shot_tdoa = []
    shot_energy = []
    shot_dr = []
    shot_x = []
    shot_y = []
    
    for p in range(n_shots_per_station):
        res = ol.capture_pulsed_toa_frame(
            pulse_width_ms=10.0,
            packet_samples=16384,
            f_target=f0,
            mic_distance_m=mic_distance_m,
            calibrated_offset_m1_ms=cal_off1_ms,
            calibrated_offset_m2_ms=cal_off2_ms,
            temperature_c=temperature_c,
            gate_cycles=3,
            timeout=1.0
        )
        
        th_ph = res["theta_phase_deg"]
        th_td = res["theta_tdoa_deg"]
        th_en = res["theta_energy_deg"]
        
        shot_phase.append(th_ph)
        shot_tdoa.append(th_td)
        shot_energy.append(th_en)
        shot_dr.append(res["delta_r_cm"])
        shot_x.append(res["x_cm"])
        shot_y.append(res["y_cm"])
        
        raw_records.append({
            "station_id": s_idx + 1,
            "theta_protractor_deg": th_tgt,
            "r_radial_cm": r_fixed_cm,
            "pulse_num": p + 1,
            # Angular Inversions
            "theta_phase_deg": th_ph,
            "theta_tdoa_deg": th_td,
            "theta_energy_deg": th_en,
            # Raw Physical Observables
            "delta_r_cm": res["delta_r_cm"],
            "delta_t_ms": res["delta_t_ms"],
            "delta_phi_rad": res["delta_phi_rad"],
            "amp1_direct_mv": res["amp_direct_a0_v"] * 1000.0,
            "amp2_direct_mv": res["amp_direct_a1_v"] * 1000.0,
            "delta_amp_mv": (res["amp_direct_a1_v"] - res["amp_direct_a0_v"]) * 1000.0,
            "energy1": res["energy_direct_a0"],
            "energy2": res["energy_direct_a1"],
            "delta_energy": res["energy_direct_a1"] - res["energy_direct_a0"],
            "energy_ratio_db": 10.0 * np.log10(max(res["energy_direct_a1"], 1e-9) / max(res["energy_direct_a0"], 1e-9)),
            # Reconstructed Cartesian Coordinates
            "x_reconstructed_cm": res["x_cm"],
            "y_reconstructed_cm": res["y_cm"],
            "range_center_cm": res["distance_cm"],
            # Quality
            "smr_a0_db": res["smr_a0_db"],
            "status": res["status"]
        })
        time.sleep(0.12)
        
    # Compute statistical parameters for each method
    st_ph = KinematicAnalytics.compute_metrology_statistics(shot_phase, ground_truth=th_tgt)
    st_td = KinematicAnalytics.compute_metrology_statistics(shot_tdoa, ground_truth=th_tgt)
    st_en = KinematicAnalytics.compute_metrology_statistics(shot_energy, ground_truth=th_tgt)
    st_dr = KinematicAnalytics.compute_metrology_statistics(shot_dr)
    
    station_summaries.append({
        "station_id": s_idx + 1,
        "theta_protractor_deg": th_tgt,
        # Phase Method Stats
        "phase_mean_deg": st_ph["mean"],
        "phase_std_deg": st_ph["std"],
        "phase_sem_deg": st_ph["sem"],
        "phase_mae_deg": st_ph["mae"],
        # TDOA Method Stats
        "tdoa_mean_deg": st_td["mean"],
        "tdoa_std_deg": st_td["std"],
        "tdoa_sem_deg": st_td["sem"],
        "tdoa_mae_deg": st_td["mae"],
        # Energy Method Stats
        "energy_mean_deg": st_en["mean"],
        "energy_std_deg": st_en["std"],
        "energy_mae_deg": st_en["mae"],
        # Path Delta
        "delta_r_mean_cm": st_dr["mean"],
        "delta_r_std_mm": st_dr["std"] * 10.0,
        # Coordinates
        "x_mean_cm": float(np.mean(shot_x)),
        "y_mean_cm": float(np.mean(shot_y))
    })
    
    print(f"   ✅ Station {th_tgt:+5.1f}° | Phase: {st_ph['mean']:+5.2f}° (MAE={st_ph['mae']:4.2f}°) | TDOA: {st_td['mean']:+5.2f}° | Energy: {st_en['mean']:+5.2f}°")

print("\n" + "=" * 85)
print("📊 ALL ANGULAR STATIONS HARVESTED!")
print("=" * 85)

## 3. 3-Method Performance Showdown & Error Corridor Plot
Compares the linearity and angular uncertainty corridor for **Phase vs. TDOA vs. Energy** across the entire $[-60^\circ, +60^\circ]$ sector.

In [ ]:
# Cell 4: 3-Method Performance Comparison Plot
df_sum = pd.DataFrame(station_summaries)
targets = df_sum["theta_protractor_deg"].to_numpy()

print("=" * 88)
print(f"{'Target (°)':<12} | {'Phase (°)':<14} | {'Phase MAE':<12} | {'TDOA (°)':<14} | {'Energy (°)':<14} | {'Status'}")
print("-" * 88)
for _, row in df_sum.iterrows():
    passed = row["phase_mae_deg"] <= 4.0
    print(f"{row['theta_protractor_deg']:+5.1f}°{'':<6} | {row['phase_mean_deg']:+5.2f}° ±{row['phase_std_deg']:4.2f}° | {row['phase_mae_deg']:4.2f}°{'':<6} | {row['tdoa_mean_deg']:+5.2f}° ±{row['tdoa_std_deg']:4.2f}° | {row['energy_mean_deg']:+5.2f}°{'':<6} | {'✅ PASS' if passed else '⚠️ CORRIDOR'}")
print("-" * 88)
print(f"Overall Sector Mean Absolute Error (Phase) : {np.mean(df_sum['phase_mae_deg']):.2f}°")
print(f"Overall Sector Mean Absolute Error (TDOA)  : {np.mean(df_sum['tdoa_mae_deg']):.2f}°")
print(f"Overall Sector Mean Absolute Error (Energy): {np.mean(df_sum['energy_mae_deg']):.2f}°\n")

# Interactive Comparison Figure
fig_aoa = go.Figure()
x_ideal = np.linspace(-70, 70, 100)
fig_aoa.add_scatter(x=x_ideal, y=x_ideal + 4.0, mode="lines", line=dict(width=0.5, color="rgba(0, 255, 204, 0.2)", dash="dot"), showlegend=False)
fig_aoa.add_scatter(x=x_ideal, y=x_ideal - 4.0, mode="lines", line=dict(width=0.5, color="rgba(0, 255, 204, 0.2)", dash="dot"), fill="tonexty", fillcolor="rgba(0, 255, 204, 0.12)", name="±4.0° Quality Corridor")
fig_aoa.add_scatter(x=x_ideal, y=x_ideal, mode="lines", line=dict(color="gray", dash="dash"), name="Ideal Ground Truth (y = x)")

# Method 1: Direct Phase
fig_aoa.add_scatter(
    x=targets, y=df_sum["phase_mean_deg"],
    error_y=dict(type="data", array=df_sum["phase_std_deg"], visible=True),
    mode="lines+markers", marker=dict(size=10, color="#00FFCC"),
    line=dict(color="#00FFCC", width=2.5),
    name=f"Autodyne Phase (MAE={np.mean(df_sum['phase_mae_deg']):.2f}°)"
)

# Method 2: TDOA
fig_aoa.add_scatter(
    x=targets, y=df_sum["tdoa_mean_deg"],
    error_y=dict(type="data", array=df_sum["tdoa_std_deg"], visible=True),
    mode="lines+markers", marker=dict(size=8, color="#FFA500", symbol="diamond"),
    line=dict(color="#FFA500", width=1.8, dash="dot"),
    name=f"TDOA Threshold (MAE={np.mean(df_sum['tdoa_mae_deg']):.2f}°)"
)

# Method 3: Energy
fig_aoa.add_scatter(
    x=targets, y=df_sum["energy_mean_deg"],
    mode="lines+markers", marker=dict(size=7, color="#E040FB", symbol="triangle-up"),
    line=dict(color="#E040FB", width=1.5, dash="dash"),
    name=f"Energy Difference (MAE={np.mean(df_sum['energy_mae_deg']):.2f}°)"
)

fig_aoa.update_layout(
    template="plotly_dark", height=500,
    title="<b>Angular Metrology Benchmark: Phase vs. TDOA vs. Energy Monopulse</b>",
    xaxis=dict(title="Protractor Ground Truth Angle [°]", range=[-70, 70]),
    yaxis=dict(title="Estimated Incident Bearing Angle [°]", range=[-70, 70])
)
fig_aoa.show()

## 4. 2D Euclidean Ray Reconstruction
Reconstructs the physical acoustic ray vectors in 2D Euclidean space originating from the microphone baseline midpoint $(0, 0)$ out to the source positions.

In [ ]:
# Cell 5: 2D Acoustic Ray Visualization
r_m = r_fixed_cm / 100.0
colors = ["#FFA500", "#00E5FF", "#76FF03", "#E040FB", "#FFD600", "#FF5252", "#69F0AE", "#40C4FF", "#FFAB40"]

fig_rays = go.Figure()

# Baseline representation
fig_rays.add_scatter(
    x=[-0.025, 0.025], y=[0, 0], mode="lines+markers",
    marker=dict(size=10, color=["#00FFCC", "#FF007F"]),
    line=dict(color="white", width=4),
    name="Mic Baseline (d=5cm)"
)

for idx, row in df_sum.iterrows():
    t_tgt = row["theta_protractor_deg"]
    t_meas = row["phase_mean_deg"]
    col = colors[idx % len(colors)]
    
    # Target Ray (Dotted)
    rad_t = np.radians(t_tgt)
    fig_rays.add_scatter(
        x=[0, r_m * np.sin(rad_t)], y=[0, r_m * np.cos(rad_t)],
        mode="lines", line=dict(color=col, dash="dot", width=1.5),
        showlegend=False
    )
    # Measured Ray (Solid)
    rad_m = np.radians(t_meas)
    fig_rays.add_scatter(
        x=[0, r_m * np.sin(rad_m)], y=[0, r_m * np.cos(rad_m)],
        mode="lines+markers", line=dict(color=col, width=2.5),
        marker=dict(size=[0, 8]),
        name=f"Ray {t_tgt:+.0f}°"
    )

fig_rays.update_layout(
    template="plotly_dark", height=500,
    title="<b>2D Reconstructed Directional Acoustic Ray Beams (Autodyne Phase)</b>",
    xaxis=dict(title="Horizontal Position X [m]", range=[-0.35, 0.35]),
    yaxis=dict(title="Forward Distance Y [m]", range=[-0.03, 0.38])
)
fig_rays.show()

## 5. Multi-Tab Excel Export
Exports the entire dataset into **`data/angle_characterization_raw.xlsx`** containing:
* **Tab 1 (`Raw_Pings`):** Every individual pulse with raw $\Delta t$, $\Delta\phi$, $\Delta A$, $\Delta E$, $\theta_{\text{TDOA}}$, $\theta_{\text{phase}}$, $\theta_{\text{energy}}$, and reconstructed $(x, y)$.
* **Tab 2 (`Station_Summary`):** Statistical parameters per station ($\mu, \sigma, \text{SEM}, \text{MAE}$ for all 3 methods).
* **Tab 3 (`Method_Benchmark`):** Overall sector MAE, RMSE, Pearson correlation, and spatial resolution metrics.

In [ ]:
# Cell 6: Multi-Tab Excel Exporter
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)
excel_path = data_dir / "angle_characterization_raw.xlsx"
csv_path = data_dir / "angle_characterization_raw.csv"

df_raw = pd.DataFrame(raw_records)
df_summary = pd.DataFrame(station_summaries)

# Overall comparative benchmark sheet
df_benchmark = pd.DataFrame([{
    "carrier_freq_hz": f0,
    "mic_baseline_cm": mic_distance_m * 100.0,
    "radial_arc_cm": r_fixed_cm,
    "temperature_c": temperature_c,
    "speed_of_sound_mps": c_sound,
    "phase_overall_mae_deg": float(np.mean(df_summary["phase_mae_deg"])),
    "phase_overall_rmse_deg": float(np.sqrt(np.mean(df_summary["phase_mae_deg"] ** 2))),
    "tdoa_overall_mae_deg": float(np.mean(df_summary["tdoa_mae_deg"])),
    "tdoa_overall_rmse_deg": float(np.sqrt(np.mean(df_summary["tdoa_mae_deg"] ** 2))),
    "energy_overall_mae_deg": float(np.mean(df_summary["energy_mae_deg"])),
    "energy_overall_rmse_deg": float(np.sqrt(np.mean(df_summary["energy_mae_deg"] ** 2))),
    "theoretical_sample_resolution_deg": float(np.degrees(np.arcsin(c_sound / ol.fs_per_ch / mic_distance_m)))
}])

# Write Excel Workbook
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_raw.to_excel(writer, sheet_name="Raw_Pings", index=False)
    df_summary.to_excel(writer, sheet_name="Station_Summary", index=False)
    df_benchmark.to_excel(writer, sheet_name="Method_Benchmark", index=False)

# Save raw CSV backup
df_raw.to_csv(csv_path, index=False)

print("=" * 85)
print("🎉 ANGULAR DATA EXPORT COMPLETE!")
print("=" * 85)
print(f"• Multi-Tab Excel Workbook : {excel_path.resolve()}")
print(f"• Raw Telemetry CSV Backup : {csv_path.resolve()}")
print("=" * 85)

## 6. Clean Hardware Teardown
Release FPGA DMA channels and hardware memory buffers.

In [ ]:
ol.close()
print("🔒 FPGA hardware resources cleanly released.")